# Polling trends

Per-party 7-day rolling mean from the GB national-VI poll table. Sanity-checks the data engine output.

In [ ]:
import os
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repo root (no pyproject.toml in cwd or any parent)")

os.chdir(_find_repo_root())

def _latest_snapshot_hash() -> str:
    """Return the content hash of the most recent snapshot. Filenames are
    YYYY-MM-DD__v<schema>__<hash>.sqlite, so lexical sort = chronological."""
    snaps = sorted(Path("data/snapshots").glob("*.sqlite"))
    if not snaps:
        raise FileNotFoundError("no snapshots in data/snapshots/; run /snapshot first")
    return snaps[-1].stem.split("__")[-1]

def _pick_prediction(strategy_marker: str, label: str) -> Path:
    """Select the prediction file for the LATEST snapshot whose name contains
    strategy_marker AND label. Prediction filenames are
    <snap_hash>__<strategy>__<config_hash>__<label>.sqlite. Fails loud if 0 or
    >1 files match — the previous sorted(glob)[-1] silently picked an
    alphabetically-last file, which became nondeterministic once sweep_* runs
    or older snapshots' predictions shared the directory."""
    snap_hash = _latest_snapshot_hash()
    pred_dir = Path("data/predictions")
    matches = [
        p for p in pred_dir.glob(f"{snap_hash}__*{strategy_marker}*.sqlite")
        if label in p.name
    ]
    if not matches:
        raise FileNotFoundError(
            f"no prediction in {pred_dir} for snapshot {snap_hash} matching "
            f"strategy={strategy_marker!r} label={label!r}; run /predict first"
        )
    if len(matches) > 1:
        names = sorted(p.name for p in matches)
        raise RuntimeError(
            f"multiple predictions for snapshot {snap_hash} match "
            f"strategy={strategy_marker!r} label={label!r}: {names}; "
            f"remove duplicates or pass a more specific label"
        )
    return matches[0]

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from prediction_engine.snapshot_loader import Snapshot
from prediction_engine.analysis.poll_trends import rolling_trend

snap_path = sorted(Path("data/snapshots").glob("*.sqlite"))[-1]
snap = Snapshot(snap_path)
trend = rolling_trend(snap, window_days=7, geography="GB")
trend.tail()

In [ ]:
ax = trend.plot(figsize=(10, 5))
ax.set_ylabel("Vote share (%)")
ax.set_title(f"7-day rolling per-party national VI trend (as of {snap.manifest.as_of_date})")
plt.show()

Lines should be smooth (no sub-cell spikes); Reform should sit above other parties when the snapshot's GB national VI shows it leading.